In [ ]:
#preprocessing nltk 1996-99
import pandas as pd
import os
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from unidecode import unidecode
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(message)s')

# Define corrections for common encoding errors and final spelling adjustments
corrections = {
    "ladi": "lady",
    "gentlemeni": "gentlemen",
    "presidi": "president",
    "happi": "happy",
    # Add more common misrepresentations as needed
}

def preprocess_text_standard(text):
    """ Apply full preprocessing: remove symbols, lowercase, remove stopwords, and tokenize. """
    if pd.isna(text):
        return ""
    
    # Convert accents to ASCII equivalents
    text = unidecode(text)
    
    # Tokenize text
    tokens = word_tokenize(text)
    
    # Lowercase all tokens
    tokens = [word.lower() for word in tokens]
    
    # Remove stopwords and punctuation
    stop_words = set(stopwords.words("english"))
    tokens = [word for word in tokens if word not in stop_words and re.match(r'\w+', word)]
    
    return " ".join(tokens)

def apply_targeted_artifact_removal(text):
    """ After standard preprocessing, remove specific artifacts like â€“, â€, etc. """
    replacements = {
        "â€“": "-",   # en-dash artifact
        "â€”": "-",   # em-dash artifact
        "â€™": "'",   # right single quote / apostrophe
        "â€œ": '"', "â€": '"',  # double quotes
        "â€˜": "'",   # left single quote
        "â€¦": "...", # ellipsis
        "â€": "",     # any remaining quotation artifact
        "â‚¬": "€"    # Euro symbol
    }
    for wrong, correct in replacements.items():
        text = text.replace(wrong, correct)
    
    return text

def apply_final_corrections(text):
    """ Apply specific corrections for words ending in y (e.g., ladi -> lady) after NLTK processing. """
    for wrong, correct in corrections.items():
        text = re.sub(rf'\b{wrong}\b', correct, text)
    return text

def final_cleanup(text):
    """ Final cleanup to remove any remaining non-alphanumeric symbols or punctuation. """
    return re.sub(r"[^\w\s]", "", text)  # Remove all non-alphanumeric characters

def contains_encoding_artifacts(text, artifact_summary):
    """ Check for remaining 'weird' symbols like â€ and log them. """
    artifacts = re.findall(r'[^\x00-\x7F]', text)
    if artifacts:
        for artifact in artifacts:
            artifact_summary[artifact] = artifact_summary.get(artifact, 0) + 1
        return True
    return False

def process_csv_files(file_paths):
    for file_path in file_paths:
        # Extract the year from the folder structure and set up output directory
        year_folder = os.path.basename(os.path.dirname(file_path))
        year = year_folder.split("_")[0]  # Assumes format like '1996_country_for_nltk'
        parent_folder = os.path.dirname(os.path.dirname(file_path))  # Go one level up
        output_folder = os.path.join(parent_folder, f"{year}_preprocessed")
        os.makedirs(output_folder, exist_ok=True)
        
        # Define the output file path
        output_file = os.path.join(output_folder, f"{year}_preprocessed.csv")
        
        # Load the CSV file
        df = pd.read_csv(file_path)
        
        # Track foreign character rows count and encoding artifact summary
        foreign_character_rows = 0
        total_rows = len(df)
        encoding_artifact_summary = {}

        # Remove 'Annex_Form of Speech' and 'Annex_Topic' columns if they exist
        for column in ['Annex_Form of Speech', 'Annex_Topic']:
            if column in df.columns:
                df.drop(columns=[column], inplace=True)
        
        # Apply symbol removal to specified columns without changing case or lemmatizing/stemming
        for column in ['Annex_Speaker', 'Annex_Party', 'Annex_Country']:
            if column in df.columns:
                df[column] = df[column].fillna("").apply(unidecode).apply(remove_symbols)
        
        # Apply full text preprocessing with encoding fix to specified columns
        for column in ['Quote', 'Annex_Quote']:
            if column in df.columns:
                # Apply standard preprocessing
                df[column] = df[column].fillna("").apply(preprocess_text_standard)
                # Apply targeted artifact removal to handle specific artifacts after standard preprocessing
                df[column] = df[column].apply(apply_targeted_artifact_removal)
                # Apply final corrections for specific words like 'ladi' -> 'lady'
                df[column] = df[column].apply(apply_final_corrections)
                # Apply final cleanup to remove any lingering symbols
                df[column] = df[column].apply(final_cleanup)
                # Check for encoding artifacts
                foreign_character_rows += df[column].apply(lambda x: contains_encoding_artifacts(x, encoding_artifact_summary)).sum()

        # Save the processed DataFrame
        df.to_csv(output_file, index=False)
        
        # Log the processing summary
        logging.info(f"Processed file saved at: {output_file}")
        logging.info(f"Total rows processed: {total_rows}")
        logging.info(f"Rows with encoding artifacts or non-ASCII characters: {foreign_character_rows}")
        
        # Print summary of unique encoding artifacts found
        if encoding_artifact_summary:
            logging.info("Summary of encoding artifacts found:")
            for artifact, count in encoding_artifact_summary.items():
                logging.info(f"  '{artifact}': {count} occurrences")
        else:
            logging.info("No encoding artifacts detected.")

# Example usage
file_paths = [
    r"C:\Users\pablo\OneDrive\Escritorio\Masters\Year_2\eu_database_project\preprocessing\1994-1999_CRE-4\1996\1996_country_for_nltk\1996_country_for_nltk.csv",
    r"C:\Users\pablo\OneDrive\Escritorio\Masters\Year_2\eu_database_project\preprocessing\1994-1999_CRE-4\1997\1997_country_for_nltk\1997_country_for_nltk.csv",
    r"C:\Users\pablo\OneDrive\Escritorio\Masters\Year_2\eu_database_project\preprocessing\1994-1999_CRE-4\1998\1998_country_for_nltk\1998_country_for_nltk.csv",
    r"C:\Users\pablo\OneDrive\Escritorio\Masters\Year_2\eu_database_project\preprocessing\1994-1999_CRE-4\1999\1999_country_for_nltk\1999_country_for_nltk.csv",
]
process_csv_files(file_paths)


In [ ]:
#preprocessing nltk 1999-2024 (With data already translated)
import pandas as pd
import os
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from unidecode import unidecode
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s', datefmt='%Y-%m-%d %H:%M:%S')

# Define corrections for common encoding errors and final spelling adjustments
corrections = {
    "ladi": "lady",
    "gentlemeni": "gentlemen",
    "presidi": "president",
    "happi": "happy",
    # Add more common misrepresentations as needed
}

def preprocess_text_standard(text):
    """ Apply full preprocessing: remove symbols, lowercase, remove stopwords, and tokenize. """
    if pd.isna(text):
        return ""
    
    # Convert accents to ASCII equivalents
    text = unidecode(text)
    
    # Tokenize text
    tokens = word_tokenize(text)
    
    # Lowercase all tokens
    tokens = [word.lower() for word in tokens]
    
    # Remove stopwords and punctuation
    stop_words = set(stopwords.words("english"))
    tokens = [word for word in tokens if word not in stop_words and re.match(r'\w+', word)]
    
    return " ".join(tokens)

def apply_targeted_artifact_removal(text):
    """ After standard preprocessing, remove specific artifacts like â€“, â€, etc. """
    replacements = {
        "â€“": "-",   # en-dash artifact
        "â€”": "-",   # em-dash artifact
        "â€™": "'",   # right single quote / apostrophe
        "â€œ": '"', "â€": '"',  # double quotes
        "â€˜": "'",   # left single quote
        "â€¦": "...", # ellipsis
        "â€": "",     # any remaining quotation artifact
        "â‚¬": "€"    # Euro symbol
    }
    for wrong, correct in replacements.items():
        text = text.replace(wrong, correct)
    
    return text

def apply_final_corrections(text):
    """ Apply specific corrections for words ending in y (e.g., ladi -> lady) after NLTK processing. """
    for wrong, correct in corrections.items():
        text = re.sub(rf'\b{wrong}\b', correct, text)
    return text

def final_cleanup(text):
    """ Final cleanup to remove any remaining non-alphanumeric symbols or punctuation. """
    return re.sub(r"[^\w\s]", "", text)  # Remove all non-alphanumeric characters

def contains_encoding_artifacts(text, artifact_summary):
    """ Check for remaining 'weird' symbols like â€ and log them. """
    artifacts = re.findall(r'[^\x00-\x7F]', text)
    if artifacts:
        for artifact in artifacts:
            artifact_summary[artifact] = artifact_summary.get(artifact, 0) + 1
        return True
    return False

def process_csv_files_in_place(directory_path):
    # Target column names to preprocess
    target_columns = ["Quote"]
    
    for file_name in os.listdir(directory_path):
        file_path = os.path.join(directory_path, file_name)

        # Process only CSV files
        if not file_name.endswith(".csv"):
            logging.info(f"Skipping non-CSV file: {file_name}")
            continue

        logging.info(f"Processing file: {file_name}")

        # Load the CSV file
        try:
            df = pd.read_csv(file_path)
        except Exception as e:
            logging.error(f"Failed to read {file_name}: {e}")
            continue

        # Log initial information about the DataFrame
        logging.info(f"File {file_name} loaded successfully with {df.shape[0]} rows and {df.shape[1]} columns.")

        # Track encoding artifact summary
        encoding_artifact_summary = {}
        columns_updated = []

        # Apply preprocessing only to specified target columns if they exist in the DataFrame
        for column_name in target_columns:
            if column_name in df.columns:
                logging.info(f"Processing column: {column_name}")
                initial_non_empty = df[column_name].notna().sum()
                
                # Apply full preprocessing on the specified column only
                df[column_name] = (
                    df[column_name]
                    .fillna("")
                    .apply(preprocess_text_standard)
                    .apply(apply_targeted_artifact_removal)
                    .apply(apply_final_corrections)
                    .apply(final_cleanup)
                )
                
                # Log changes made to the column
                final_non_empty = df[column_name].notna().sum()
                columns_updated.append(column_name)
                logging.info(f"Updated column '{column_name}': {initial_non_empty} -> {final_non_empty} non-empty rows.")
                
                # Check for encoding artifacts
                artifact_rows = df[column_name].apply(lambda x: contains_encoding_artifacts(x, encoding_artifact_summary)).sum()
                if artifact_rows > 0:
                    logging.info(f"Encoding artifacts detected in column '{column_name}': {artifact_rows} rows affected.")

        # Save the processed DataFrame back to the original file
        try:
            df.to_csv(file_path, index=False)
            logging.info(f"File saved: {file_path}")
        except Exception as e:
            logging.error(f"Failed to save {file_name}: {e}")
            continue
        
        # Log summary of processed columns
        if columns_updated:
            logging.info(f"Columns updated in {file_name}: {', '.join(columns_updated)}")
        else:
            logging.warning(f"No target columns found in {file_name}. No changes applied.")

        # Print summary of unique encoding artifacts found
        if encoding_artifact_summary:
            logging.info("Summary of encoding artifacts found:")
            for artifact, count in encoding_artifact_summary.items():
                logging.info(f"  '{artifact}': {count} occurrences")
        else:
            logging.info("No encoding artifacts detected.")

        logging.info(f"Finished processing file: {file_name}\n")

# Example usage
directory_path = r"C:\Users\pablo\OneDrive\Escritorio\Masters\Year_2\eu_database_project\final_preprocessed_data_2"
process_csv_files_in_place(directory_path)


In [ ]:
# More stopwords and lemmatization
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import pandas as pd
import os

# Define your custom stopwords here
custom_stopwords = set(stopwords.words('english'))
additional_stopwords = {
     'a',
    'about',
    'above',
    'across',
    'after',
    'afterwards',
    'again',
    'against',
    'ain',
    'all',
    'almost',
    'alone',
    'along',
    'already',
    'also',
    'although',
    'always',
    'am',
    'among',
    'amongst',
    'amoungst',
    'amount',
    'an',
    'and',
    'another',
    'any',
    'anyhow',
    'anyone',
    'anything',
    'anyway',
    'anywhere',
    'are',
    'aren',
    'around',
    'as',
    'at',
    'back',
    'be',
    'became',
    'because',
    'become',
    'becomes',
    'becoming',
    'been',
    'before',
    'beforehand',
    'behind',
    'being',
    'below',
    'beside',
    'besides',
    'between',
    'beyond',
    'bill',
    'both',
    'bottom',
    'but',
    'by',
    'call',
    'can',
    'cannot',
    'cant',
    'co',
    'con',
    'could',
    'couldn',
    'couldnt',
    'cry',
    'd',
    'de',
    'describe',
    'detail',
    'did',
    'didn',
    'do',
    'does',
    'doesn',
    'doing',
    'don',
    'done',
    'down',
    'due',
    'during',
    'each',
    'eg',
    'eight',
    'either',
    'eleven',
    'else',
    'elsewhere',
    'empty',
    'enough',
    'etc',
    'even',
    'ever',
    'every',
    'everyone',
    'everything',
    'everywhere',
    'except',
    'few',
    'fifteen',
    'fify',
    'fill',
    'find',
    'fire',
    'first',
    'five',
    'for',
    'former',
    'formerly',
    'forty',
    'found',
    'four',
    'from',
    'front',
    'full',
    'further',
    'get',
    'give',
    'go',
    'had',
    'hadn',
    'has',
    'hasn',
    'hasnt',
    'have',
    'haven',
    'having',
    'he',
    'hence',
    'her',
    'here',
    'hereafter',
    'hereby',
    'herein',
    'hereupon',
    'hers',
    'herself',
    'him',
    'himself',
    'his',
    'how',
    'however',
    'hundred',
    'i',
    'ie',
    'if',
    'in',
    'inc',
    'indeed',
    'interest',
    'into',
    'is',
    'isn',
    'it',
    'its',
    'itself',
    'just',
    'keep',
    'last',
    'latter',
    'latterly',
    'least',
    'less',
    'll',
    'ltd',
    'm',
    'ma',
    'made',
    'many',
    'may',
    'me',
    'meanwhile',
    'might',
    'mightn',
    'mill',
    'mine',
    'more',
    'moreover',
    'most',
    'mostly',
    'move',
    'much',
    'must',
    'mustn',
    'my',
    'myself',
    'name',
    'namely',
    'needn',
    'neither',
    'never',
    'nevertheless',
    'next',
    'nine',
    'no',
    'nobody',
    'none',
    'noone',
    'nor',
    'not',
    'nothing',
    'now',
    'nowhere',
    'o',
    'of',
    'off',
    'often',
    'on',
    'once',
    'one',
    'only',
    'onto',
    'or',
    'other',
    'others',
    'otherwise',
    'our',
    'ours',
    'ourselves',
    'out',
    'over',
    'own',
    'part',
    'per',
    'perhaps',
    'please',
    'put',
    'rather',
    're',
    's',
    'same',
    'see',
    'seem',
    'seemed',
    'seeming',
    'seems',
    'serious',
    'several',
    'shan',
    'she',
    'should',
    'shouldn',
    'show',
    'side',
    'since',
    'sincere',
    'six',
    'sixty',
    'so',
    'some',
    'somehow',
    'someone',
    'something',
    'sometime',
    'sometimes',
    'somewhere',
    'still',
    'such',
    'system',
    't',
    'take',
    'ten',
    'than',
    'that',
    'the',
    'their',
    'theirs',
    'them',
    'themselves',
    'then',
    'thence',
    'there',
    'thereafter',
    'thereby',
    'therefore',
    'therein',
    'thereupon',
    'these',
    'they',
    'thick',
    'thin',
    'third',
    'this',
    'those',
    'though',
    'three',
    'through',
    'throughout',
    'thru',
    'thus',
    'to',
    'together',
    'too',
    'top',
    'toward',
    'towards',
    'twelve',
    'twenty',
    'two',
    'un',
    'under',
    'until',
    'up',
    'upon',
    'us',
    've',
    'very',
    'via',
    'was',
    'wasn',
    'we',
    'well',
    'were',
    'weren',
    'what',
    'whatever',
    'when',
    'whence',
    'whenever',
    'where',
    'whereafter',
    'whereas',
    'whereby',
    'wherein',
    'whereupon',
    'wherever',
    'whether',
    'which',
    'while',
    'whither',
    'who',
    'whoever',
    'whole',
    'whom',
    'whose',
    'why',
    'will',
    'with',
    'within',
    'without',
    'won',
    'would',
    'wouldn',
    'y',
    'yet',
    'you',
    'your',
    'yours',
    'yourself',
    'yourselves'
}
custom_stopwords.update(additional_stopwords)

lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    words = text.split()
    lemmatized_words = []
    stopwords_removed = 0
    
    for word in words:
        if word.lower() in custom_stopwords:
            stopwords_removed += 1
        else:
            lemmatized_words.append(lemmatizer.lemmatize(word.lower()))
    
    return ' '.join(lemmatized_words), stopwords_removed

def process_files(directory):
    total_stopwords_removed = 0
    
    for filename in os.listdir(directory):
        if filename.endswith('.csv'):
            filepath = os.path.join(directory, filename)
            df = pd.read_csv(filepath)
            
            for column in ['Quote', 'Subject', 'Questions', 'Answers', 'Annex_Quote']:
                if column in df.columns:
                    stopwords_removed_column = 0
                    processed_texts = []
                    
                    for text in df[column].dropna():
                        processed_text, stopwords_removed = preprocess_text(text)
                        processed_texts.append(processed_text)
                        stopwords_removed_column += stopwords_removed
                    
                    # Update the column with processed texts
                    df[column] = pd.Series(processed_texts, index=df[column].dropna().index)
                    total_stopwords_removed += stopwords_removed_column
            
            # Save the processed file
            df.to_csv(filepath, index=False)
    
    print(f"Total stopwords removed: {total_stopwords_removed}")

# Usage
directory_path = r"C:\Users\pablo\OneDrive\Escritorio\Masters\Year_2\eu_database_project\final_preprocessed_data_2"
process_files(directory_path)
